# SHNL performance viz

Notebook učitava `shnl_full_metrics.csv`, čisti metrike i crta interaktivne grafove za usporedbu klubova. [web:111][web:118]


In [15]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

CSV_PATH = Path('shnl_full_metrics.csv')
if not CSV_PATH.exists():
    CSV_PATH = Path('output/shnl_full_metrics.csv')
df = pd.read_csv(CSV_PATH)
df.head()


,club,website,host,resolved_ip,provider,country,asn,whois_excerpt,curl_status,curl_error,...,psi_desktop_field_cls_p75,psi_desktop_field_inp_p75_ms,psi_desktop_field_fcp_p75_ms,psi_desktop_field_ttfb_p75_ms,psi_desktop_field_lcp_category,psi_desktop_field_cls_category,psi_desktop_field_inp_category,psi_desktop_origin_lcp_p75_ms,psi_desktop_origin_cls_p75,psi_desktop_origin_inp_p75_ms
0,Dinamo,https://gnkdinamo.hr/,gnkdinamo.hr,172.67.222.17,"Cloudflare, Inc.",US,13335,"{""ip"": ""172.67.222.17"", ""success"": true, ""type...",ok,NaN,...,0,56,1016,662.0,FAST,FAST,FAST,1388,0,53
1,Hajduk,https://hajduk.hr/,hajduk.hr,172.67.203.172,"Cloudflare, Inc.",US,13335,"{""ip"": ""172.67.203.172"", ""success"": true, ""typ...",ok,NaN,...,1,32,687,293.0,FAST,FAST,FAST,768,2,34
2,Rijeka,https://nk-rijeka.hr/,nk-rijeka.hr,51.89.7.44,OVH SAS,DE,16276,"{""ip"": ""51.89.7.44"", ""success"": true, ""type"": ...",ok,NaN,...,1,45,1666,1120.0,FAST,FAST,FAST,1712,1,46
3,Varaždin,https://nk-varazdin.hr/,nk-varazdin.hr,172.67.164.173,"Cloudflare, Inc.",US,13335,"{""ip"": ""172.67.164.173"", ""success"": true, ""typ...",ok,NaN,...,0,28,3607,3044.0,SLOW,FAST,FAST,3402,0,32
4,Slaven Belupo,https://nk-slaven-belupo.hr/,nk-slaven-belupo.hr,185.58.73.254,Cyber Folks D.O.O,HR,201563,"{""ip"": ""185.58.73.254"", ""success"": true, ""type...",ok,NaN,...,45,33,952,483.0,AVERAGE,SLOW,FAST,1590,36,38


In [16]:
numeric_cols = [
    'curl_time_total_s', 'curl_ttfb_after_connect_s', 'curl_dns_s',
    'lh_fcp_ms', 'lh_lcp_ms', 'lh_tbt_ms', 'lh_cls', 'lh_tti_ms',
    'lighthouse_performance_score', 'lh_total_byte_weight_bytes',
    'lh_dom_size_elements', 'psi_mobile_field_lcp_p75_ms',
    'psi_mobile_field_cls_p75', 'psi_mobile_field_inp_p75_ms',
    'psi_desktop_field_lcp_p75_ms', 'psi_desktop_field_cls_p75',
    'psi_desktop_field_inp_p75_ms'
]
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

valid = df[df['club'].notna()].copy()
valid = valid[valid['curl_status'] == 'ok'].copy()
valid[['club','website','curl_time_total_s','lh_lcp_ms','lh_tbt_ms','lh_cls','lighthouse_performance_score']]


,club,website,curl_time_total_s,lh_lcp_ms,lh_tbt_ms,lh_cls,lighthouse_performance_score
0,Dinamo,https://gnkdinamo.hr/,0.652783,3274.25550,0.0,0.002174,82.0
1,Hajduk,https://hajduk.hr/,0.120559,4727.57204,0.0,0.010526,76.0
2,Rijeka,https://nk-rijeka.hr/,0.891471,2236.07250,9.0,0.003755,82.0
3,Varaždin,https://nk-varazdin.hr/,2.715895,2774.05650,0.0,0.003503,75.0
4,Slaven Belupo,https://nk-slaven-belupo.hr/,0.408639,6629.30578,0.0,0.362113,39.0
5,Lokomotiva,https://nklokomotiva.hr/,0.885216,24254.41100,0.0,0.321351,42.0
6,Osijek,http://nk-osijek.hr/,0.261701,4189.69350,0.0,0.017228,75.0
7,Istra 1961,https://nkistra.com/,1.008900,8110.12675,21.5,0.216469,45.0
8,Vukovar,https://hnk-vukovar1991.hr/,0.236649,5456.52500,0.0,0.211876,60.0
9,Gorica,https://hnk-gorica.hr/,0.788099,3454.49200,29.5,0.266109,57.0


## Složeni score

Niži LCP, TBT, CLS i `curl_time_total_s` su bolji, a veći Lighthouse score je bolji, pa ih normaliziramo u jedan pomoćni ranking score za grubu usporedbu. [web:120]


In [17]:
rank_df = valid.copy()
metrics = {
    'lighthouse_performance_score': 'max',
    'lh_lcp_ms': 'min',
    'lh_tbt_ms': 'min',
    'lh_cls': 'min',
    'curl_time_total_s': 'min'
}

for col, direction in metrics.items():
    s = rank_df[col].astype(float)
    min_v, max_v = s.min(), s.max()
    if pd.isna(min_v) or pd.isna(max_v) or min_v == max_v:
        rank_df[col + '_norm'] = 0.5
    else:
        if direction == 'max':
            rank_df[col + '_norm'] = (s - min_v) / (max_v - min_v)
        else:
            rank_df[col + '_norm'] = (max_v - s) / (max_v - min_v)

weights = {
    'lighthouse_performance_score_norm': 0.35,
    'lh_lcp_ms_norm': 0.30,
    'lh_tbt_ms_norm': 0.20,
    'lh_cls_norm': 0.10,
    'curl_time_total_s_norm': 0.05,
}
rank_df['overall_score'] = sum(rank_df[k] * w for k, w in weights.items())
rank_df = rank_df.sort_values('overall_score', ascending=False)
rank_df[['club','overall_score','lighthouse_performance_score','lh_lcp_ms','lh_tbt_ms','lh_cls','curl_time_total_s']]


,club,overall_score,lighthouse_performance_score,lh_lcp_ms,lh_tbt_ms,lh_cls,curl_time_total_s
0,Dinamo,0.975601,82.0,3274.25550,0.0,0.002174,0.652783
2,Rijeka,0.923692,82.0,2236.07250,9.0,0.003755,0.891471
1,Hajduk,0.914896,76.0,4727.57204,0.0,0.010526,0.120559
6,Osijek,0.909504,75.0,4189.69350,0.0,0.017228,0.261701
3,Varaždin,0.885324,75.0,2774.05650,0.0,0.003503,2.715895
8,Vukovar,0.716555,60.0,5456.52500,0.0,0.211876,0.236649
9,Gorica,0.493723,57.0,3454.49200,29.5,0.266109,0.788099
4,Slaven Belupo,0.484592,39.0,6629.30578,0.0,0.362113,0.408639
7,Istra 1961,0.396390,45.0,8110.12675,21.5,0.216469,1.008900
5,Lokomotiva,0.271012,42.0,24254.41100,0.0,0.321351,0.885216


In [8]:
fig = px.bar(
    rank_df,
    x='club', y='overall_score', color='overall_score',
    title='Ukupni performance score po klubu',
    text=rank_df['overall_score'].round(3),
    color_continuous_scale='Viridis'
)
fig.update_layout(xaxis_title='', yaxis_title='Composite score')
fig.show()


## Ključne metrike


In [18]:
key_metrics = ['lh_lcp_ms', 'lh_tbt_ms', 'lh_cls', 'curl_time_total_s', 'lighthouse_performance_score']
long_df = rank_df[['club'] + key_metrics].melt(id_vars='club', var_name='metric', value_name='value')
fig = px.bar(long_df, x='club', y='value', color='metric', barmode='group', title='Usporedba ključnih metrika')
fig.show()


In [10]:
for metric, title in [
    ('lh_lcp_ms', 'Largest Contentful Paint (ms)'),
    ('lh_tbt_ms', 'Total Blocking Time (ms)'),
    ('lh_cls', 'Cumulative Layout Shift'),
    ('curl_time_total_s', 'curl total time (s)'),
    ('lighthouse_performance_score', 'Lighthouse performance score')
]:
    fig = px.bar(rank_df.sort_values(metric, ascending=(metric!='lighthouse_performance_score')),
                 x='club', y=metric, text=metric, title=title, color=metric, color_continuous_scale='RdYlGn_r' if metric!='lighthouse_performance_score' else 'RdYlGn')
    fig.update_traces(texttemplate='%{text:.3s}')
    fig.show()


## Field data ako postoji

Ako CSV sadrži PSI/CrUX podatke, možeš posebno vizualizirati p75 LCP, CLS i INP za mobile/desktop. [web:117][web:123]


In [19]:
psi_cols = [c for c in [
    'psi_mobile_field_lcp_p75_ms', 'psi_mobile_field_cls_p75', 'psi_mobile_field_inp_p75_ms',
    'psi_desktop_field_lcp_p75_ms', 'psi_desktop_field_cls_p75', 'psi_desktop_field_inp_p75_ms'
] if c in rank_df.columns]
psi_df = rank_df[['club'] + psi_cols].copy()
psi_df


,club,psi_mobile_field_lcp_p75_ms,psi_mobile_field_cls_p75,psi_mobile_field_inp_p75_ms,psi_desktop_field_lcp_p75_ms,psi_desktop_field_cls_p75,psi_desktop_field_inp_p75_ms
0,Dinamo,1539,0,138,1574,0,56
2,Rijeka,2050,1,93,2009,1,45
1,Hajduk,1098,0,86,913,1,32
6,Osijek,1564,2,78,1625,6,33
3,Varaždin,4234,0,84,4122,0,28
8,Vukovar,2872,4,99,2832,70,45
9,Gorica,2735,12,307,4248,42,302
4,Slaven Belupo,1996,2,86,3521,45,33
7,Istra 1961,4006,28,98,3944,20,59
5,Lokomotiva,4863,0,136,5812,24,81


In [20]:
if psi_cols:
    long_psi = psi_df.melt(id_vars='club', var_name='metric', value_name='value').dropna()
    fig = px.bar(long_psi, x='club', y='value', color='metric', barmode='group', title='PSI / CrUX field metrike')
    fig.show()


## Scatter pogled

Koristan graf je LCP naspram TBT uz veličinu bubblea po `curl_time_total_s`, jer brzo pokaže koje stranice pate od frontenda, a koje od mreže. [web:120]


In [21]:
fig = px.scatter(
    rank_df,
    x='lh_lcp_ms', y='lh_tbt_ms', size='curl_time_total_s', color='club',
    hover_data=['lh_cls', 'lighthouse_performance_score', 'website'],
    title='LCP vs TBT (veličina mjehura = curl total time)'
)
fig.show()


## Export tablica za daljnju obradu


In [14]:
rank_out = rank_df[['club','website','overall_score','lighthouse_performance_score','lh_lcp_ms','lh_tbt_ms','lh_cls','curl_time_total_s']].copy()
rank_out.to_csv('shnl_ranked_metrics.csv', index=False)
rank_out


,club,website,overall_score,lighthouse_performance_score,lh_lcp_ms,lh_tbt_ms,lh_cls,curl_time_total_s
2,Rijeka,https://nk-rijeka.hr/,0.981816,80.0,2325.608000,2.0,0.003486,0.960624
0,Dinamo,https://gnkdinamo.hr/,0.976907,80.0,3734.213500,0.0,0.001932,0.238744
1,Hajduk,https://hajduk.hr/,0.931512,77.0,4880.904320,0.0,0.010838,0.125408
6,Osijek,http://nk-osijek.hr/,0.918705,75.0,4274.664000,0.0,0.014056,0.220421
3,Varaždin,https://nk-varazdin.hr/,0.885511,74.0,2781.070000,0.0,0.004005,2.800261
4,Slaven Belupo,https://nk-slaven-belupo.hr/,0.748891,65.0,2961.140000,0.0,0.355594,0.125333
7,Istra 1961,https://nkistra.com/,0.506802,45.0,8060.915319,0.0,0.216444,1.091917
5,Lokomotiva,https://nklokomotiva.hr/,0.473917,58.0,23010.497500,0.0,0.000000,1.086978
9,Gorica,https://hnk-gorica.hr/,0.338904,43.0,3844.113500,251.5,0.266704,0.878072
